In [1]:
import librosa
import numpy as np
import soundfile as sf
import os
import random
import glob

class BabyCryScientificAugmenter:
    def __init__(self, sr=16000):
        self.sr = 16000
        self.transformations = {
            "stretch": lambda y: librosa.effects.time_stretch(y, rate=random.uniform(0.8, 1.2)),
            "pitch": lambda y: librosa.effects.pitch_shift(y, sr=self.sr, n_steps=random.uniform(-3, 3)),
            "shift": lambda y: np.roll(y, int(random.uniform(-0.2, 0.2) * self.sr)),
            "invert": lambda y: -y
        }

    def _apply_random_aug(self, y):
        """Áp dụng ngẫu nhiên 1 hoặc 2 kỹ thuật (theo Alam 2025)"""
        num_to_apply = random.randint(1, 2)
        keys = random.sample(list(self.transformations.keys()), num_to_apply)
        
        y_aug = y.copy()
        for key in keys:
            y_aug = self.transformations[key](y_aug)
        return y_aug

    def balance_with_95_rule(self, root_dir, output_dir):
        # 1. Xác định số lượng mẫu của lớp Hungry làm chuẩn
        # Theo Donate-a-Cry, Hungry thường có 382 mẫu
        hungry_files = glob.glob(os.path.join(root_dir, "hungry", "*.wav"))
        n_hungry = len(hungry_files)
        if n_hungry == 0:
            print("Lỗi: Không tìm thấy thư mục 'hungry' hoặc file .wav")
            return

        # Tính tổng mục tiêu cho tất cả các lớp khác (95% của Hungry)
        target_total_others = int(n_hungry * 0.95) 
        
        # 2. Lấy danh sách các lớp thiểu số
        all_classes = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]
        minority_classes = [c for c in all_classes if c.lower() != "hungry"]
        n_min_classes = len(minority_classes)
        
        # Số lượng mẫu mục tiêu cho mỗi lớp thiểu số (chia đều)
        samples_per_class_target = target_total_others // n_min_classes

        print(f"--- BÁO CÁO CẤU HÌNH ---")
        print(f"Hungry (Gốc): {n_hungry} file")
        print(f"Tổng mục tiêu các lớp khác (95%): {target_total_others} file")
        print(f"Mỗi lớp thiểu số cần đạt: {samples_per_class_target} file")
        print(f"------------------------\n")

        for cls in all_classes:
            src_path = os.path.join(root_dir, cls)
            dst_path = os.path.join(output_dir, cls)
            os.makedirs(dst_path, exist_ok=True)
            
            orig_files = glob.glob(os.path.join(src_path, "*.wav"))
            n_orig = len(orig_files)

            # Copy file gốc sang folder mới
            for f in orig_files:
                y, _ = librosa.load(f, sr=self.sr)
                sf.write(os.path.join(dst_path, os.path.basename(f)), y, self.sr)

            # Chỉ tăng cường nếu không phải lớp Hungry
            if cls.lower() != "hungry":
                needed = samples_per_class_target - n_orig
                if needed > 0:
                    print(f"Đang xử lý lớp [{cls}]: Gốc {n_orig} -> Tạo thêm {needed}")
                    for i in range(needed):
                        # Chọn ngẫu nhiên 1 file gốc để biến đổi
                        source_f = random.choice(orig_files)
                        y, _ = librosa.load(source_f, sr=self.sr)
                        
                        # Áp dụng tổ hợp ngẫu nhiên
                        y_aug = self._apply_random_aug(y)
                        
                        # Lưu với tên mới để phân biệt
                        new_name = f"aug_{i}_{os.path.basename(source_f)}"
                        sf.write(os.path.join(dst_path, new_name), y_aug, self.sr)
                else:
                    print(f"Lớp [{cls}] đã đủ số lượng, không tăng cường.")
            else:
                print(f"Lớp [hungry] được giữ nguyên 100% dữ liệu gốc.")

# --- THIẾT LẬP ---
# Đường dẫn thư mục chứa các folder con: hungry, belly_pain, burping, discomfort, tired
ROOT_DATA = r"C:\Users\Hoannd\Downloads\ail\git_project\-the-adults-cried\official_project_v1\ail\data\corpus\donatecry_copus_split_data\train" 
OUTPUT_DATA = r"C:\Users\Hoannd\Downloads\ail\git_project\-the-adults-cried\official_project_v1\ail\data\corpus\donatecry_copus_split_data\train_augumentation"

balancer = BabyCryScientificAugmenter(sr=16000)
balancer.balance_with_95_rule(ROOT_DATA, OUTPUT_DATA)

--- BÁO CÁO CẤU HÌNH ---
Hungry (Gốc): 305 file
Tổng mục tiêu các lớp khác (95%): 289 file
Mỗi lớp thiểu số cần đạt: 72 file
------------------------

Đang xử lý lớp [belly_pain]: Gốc 12 -> Tạo thêm 60
Đang xử lý lớp [burping]: Gốc 6 -> Tạo thêm 66
Đang xử lý lớp [discomfort]: Gốc 21 -> Tạo thêm 51
Lớp [hungry] được giữ nguyên 100% dữ liệu gốc.
Đang xử lý lớp [tired]: Gốc 19 -> Tạo thêm 53
